# Get to Know a Dataset: GeoNadir FAIR Data

This notebook is a guided tour of the [GeoNadir FAIR Data](https://registry.opendata.aws/geonadir-fair-data) collection: open UAV (drone) survey datasets contributed through GeoNadir and published as cloud-native geospatial files. You can also browse the collection in your browser at [https://data.geonadir.com/fairgeo](https://data.geonadir.com/fairgeo). More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

> **Access method 2 of 2: the STAC catalog.** This notebook discovers and addresses data purely through the GeoNadir FAIR Data **STAC API** (`pystac-client`): you search by area of interest, date, and habitat, and STAC hands back each survey's metadata and direct asset links. It never lists or `get_object`s the bucket by hand — there is no `boto3` here. The only AWS-level step is reading the Cloud Optimized GeoTIFF *pixels* over `s3://` with Rasterio (unsigned, public).
>
> The companion notebook, `get-to-know-a-dataset.ipynb`, is **access method 1 of 2: plain AWS S3** — `boto3` only, no STAC — useful before the STAC endpoint is live, or if you prefer not to depend on a STAC client.

> **Draft status.** Cells that only need Rasterio + the public bucket are marked **[RUNS AT LAUNCH]**. Cells that need the STAC API online are marked **[NEEDS STAC]**.

## Understanding the dataset structure

GeoNadir FAIR Data is organized so the files are simple to address directly. Each UAV survey is one folder under the `datasets/` prefix, keyed by a stable public item id:

```text
s3://geonadir-fair-data/
  stac/                         <- STAC catalog (optional discovery layer)
  datasets/
    {item_id}/                          <- item_id = {dataset_id}-{uav_uid}
      {dataset_id}-metadata.json             (required)
      {dataset_id}-raw_images.zip            (required)
      {dataset_id}-rgb_ortho.tif           (optional, Cloud Optimized GeoTIFF)
      {dataset_id}-dsm.tif                   (optional, Cloud Optimized GeoTIFF)
      {dataset_id}-dtm.tif                   (optional, Cloud Optimized GeoTIFF)
      {dataset_id}-multispec_ortho.tif  (optional, Cloud Optimized GeoTIFF)
```

One UAV survey = one `datasets/{item_id}/` folder. Only the files that exist for a survey are present, so a survey with no DTM simply has no `{dataset_id}-dtm.tif`.

The STAC catalog under `stac/` is the discovery layer used in this notebook — you search it and it hands back direct links to these files, so you never have to browse the bucket by hand.

In [ ]:
# CODING GUIDELINES (from the AWS template):
# This is a 101-level introduction using common tools and libraries.
# Discovery here goes through STAC only (pystac-client). The bucket is public,
# so the COG pixel reads over s3:// are UNSIGNED (no AWS account or credentials needed).

First, the libraries. Discovery goes through the STAC API (`pystac-client`); we open Cloud Optimized GeoTIFFs directly from `s3://` with Rasterio. The dataset is public, so those COG reads are unsigned. Note there is no `boto3` in this notebook — the direct-S3 access lives in the plain-S3 companion.

In [ ]:
# This notebook requires the following additional libraries
# (install with pip or conda for your environment):
#
# pystac-client >= 0.7
# rasterio >= 1.3
# matplotlib >= 3.8
# numpy >= 1.26

# Built-ins
from pprint import pprint

# Installed libraries
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import rasterio.session
from pystac_client import Client

Next, the dataset location. Update these if the final published names differ from the draft.

In [ ]:
# [RUNS AT LAUNCH] Dataset location
STAC_API = "https://data.geonadir.com/stac"
COLLECTION = "geonadir-fair-data"

# Public bucket -> unsigned reads for the COG pixels (no credentials required).
# This AWSSession is the ONLY AWS-level access in this notebook; everything
# else — search, metadata, asset links — comes from STAC.
aws_session = rasterio.session.AWSSession(aws_unsigned=True)  # for reading COGs from s3://

With STAC we can ask for exactly the surveys we want — by area of interest, date range, and habitat — instead of listing the bucket.

In [ ]:
# [NEEDS STAC] Open the catalog and search
catalog = Client.open(STAC_API)
search = catalog.search(
    collections=[COLLECTION],
    datetime="2024-01-01T00:00:00Z/2024-12-31T23:59:59Z",
    intersects={
        "type": "Polygon",
        "coordinates": [[[151.0, -34.0], [151.2, -34.0], [151.2, -33.8], [151.0, -33.8], [151.0, -34.0]]],
    },
    filter="geonadir:iucn_habitat = 'Forest'",
    max_items=10,
)
items = list(search.items())
print(f"Found {len(items)} matching survey(s)")
for it in items:
    print(it.id, it.datetime, sorted(it.assets.keys()))

## Data formats

GeoNadir FAIR Data uses a small, deliberately standard set of formats so the data works with off-the-shelf tools:

| Asset | Format | Why this format |
| --- | --- | --- |
| `{dataset_id}-metadata.json` | JSON | Normalized survey metadata; readable in any language |
| `{dataset_id}-raw_images.zip` | ZIP of JPEG | Original UAV frames kept together for reprocessing / photogrammetry |
| `{dataset_id}-rgb_ortho.tif`, `{dataset_id}-dsm.tif`, `{dataset_id}-dtm.tif`, `{dataset_id}-multispec_ortho.tif` | Cloud Optimized GeoTIFF (COG) | Read a window or overview **directly from S3** without downloading the whole file |

The COG is the key choice. A Cloud Optimized GeoTIFF is a normal GeoTIFF with internal tiling + overviews arranged so an HTTP range-request pulls only the pixels you ask for. GDAL, Rasterio, QGIS, and cloud-native pipelines can open a multi-gigabyte orthomosaic and read a small window or a low-res overview in seconds. Each COG carries its own CRS and geotransform, so projection metadata is read straight from the file. Bands follow the common EO convention (`B1`, `B2`, ... with `common_name` like `red`, `green`, `blue`, `rededge`, `nir`).

## Loading data from a survey

First the survey's metadata — which STAC already handed us in the item's `properties`, no extra fetch — then the interesting part: opening an orthomosaic COG and reading only a low-resolution overview, straight from S3, without downloading the full GeoTIFF.

In [ ]:
# [NEEDS STAC] Inspect the first matching survey.
# In STAC the item IS the metadata: its properties and asset links come straight
# from the catalog, so there is no separate metadata.json fetch from S3.
item = items[0]
item_id = item.id

print("Item id: ", item_id)
print("Datetime:", item.datetime)
print("BBox:    ", item.bbox)
print("Assets:  ", sorted(item.assets.keys()))
print("\nProperties:")
pprint(item.properties)

Now the orthomosaic. We open it with Rasterio over `s3://` (GDAL's `/vsis3/`). Because it is a COG, reading a decimated overview transfers only a few tiles — not the whole file.

In [ ]:
# [NEEDS STAC] Resolve the orthomosaic href from the STAC item
ortho_href = item.assets['ortho'].href

with rasterio.Env(session=aws_session):
    with rasterio.open(ortho_href) as src:
        print('CRS:', src.crs)
        print('Size (WxH):', src.width, 'x', src.height)
        print('Bands:', src.count)
        print('Resolution (m):', src.res)
        print('Bounds:', src.bounds)
        # Read a downsampled overview: at most ~1024 px on the long edge
        scale = max(1, max(src.width, src.height) // 1024)
        out_shape = (src.count, src.height // scale, src.width // scale)
        overview = src.read(out_shape=out_shape)
print('Overview array shape (bands, rows, cols):', overview.shape)

## Visualizing a survey

Two visuals. First, the RGB orthomosaic overview we just read — the survey as an image. Second, a collection-level view: where the surveys are, colored by IUCN habitat.

In [ ]:
# [RUNS AT LAUNCH] Display the RGB orthomosaic overview
rgb = np.transpose(overview[:3], (1, 2, 0)).astype('float32')  # first 3 bands: R,G,B
rgb = (rgb - rgb.min()) / (rgb.ptp() + 1e-9)                   # normalize to 0-1 for display

plt.figure(figsize=(9, 9), dpi=100)
plt.imshow(rgb)
plt.title(f'GeoNadir FAIR Data orthomosaic overview: {item_id}', fontsize=13, fontweight='bold')
plt.axis('off')
plt.show()

### Quick preview from the lowest COG overview

A COG bakes in a pyramid of ever-smaller overviews — like the zoom levels of a web map stored inside the file. The **lowest (coarsest)** level is a tiny thumbnail: reading it pulls only a few kilobytes, the cheapest possible way to eyeball a survey before committing to a bigger read.

In [ ]:
# [NEEDS STAC] Load an RGB preview from the LOWEST (coarsest) COG overview.
# The coarsest overview is a tiny thumbnail — a few KB over the wire — so this is
# the cheapest possible look at a survey. (ortho_href came from the STAC item above.)
with rasterio.Env(session=aws_session):
    with rasterio.open(ortho_href) as src:
        factors = src.overviews(1)          # decimation factors, e.g. [2, 4, 8, 16, 32]
        print('COG overview factors:', factors or '(none — file has no pyramid)')
        if factors:
            f = factors[-1]                 # last = coarsest = lowest level
            out_h = -(-src.height // f)     # ceil div -> the overview's exact size
            out_w = -(-src.width  // f)
        else:
            f, out_h, out_w = 1, src.height, src.width
        # Reading at the overview's own size makes GDAL pull straight from that level
        preview = src.read([1, 2, 3], out_shape=(3, out_h, out_w))  # R, G, B
print(f'Lowest overview: 1/{f} scale, {out_w} x {out_h} px')

rgb = np.transpose(preview, (1, 2, 0)).astype('float32')
rgb = (rgb - rgb.min()) / (rgb.ptp() + 1e-9)          # normalize to 0-1 for display
plt.figure(figsize=(5, 5), dpi=100)
plt.imshow(rgb)
plt.title(f'Lowest-overview RGB preview: {item_id}', fontsize=11, fontweight='bold')
plt.axis('off'); plt.show()

For the collection footprint, we gather each survey's bounding-box centroid and habitat, then scatter them on a simple map — the kind of overview that tells a new user what the collection actually covers.

In [ ]:
# [NEEDS STAC] Map every published survey by centroid, colored by IUCN habitat
all_items = list(catalog.search(collections=[COLLECTION]).items())
records = [((it.bbox[0]+it.bbox[2])/2, (it.bbox[1]+it.bbox[3])/2,
            it.properties.get('geonadir:iucn_habitat', 'Unknown')) for it in all_items]

In [ ]:
# [RUNS AT LAUNCH] Plot the centroids
lons = [r[0] for r in records]; lats = [r[1] for r in records]; habitats = [r[2] for r in records]
cats = sorted(set(habitats))
cmap = {c: plt.cm.tab20(i % 20) for i, c in enumerate(cats)}
plt.figure(figsize=(12, 7), dpi=100)
for c in cats:
    xs = [lo for lo, h in zip(lons, habitats) if h == c]
    ys = [la for la, h in zip(lats, habitats) if h == c]
    plt.scatter(xs, ys, label=c, s=40, alpha=0.8, color=cmap[c])
plt.title('GeoNadir FAIR Data — published surveys by IUCN habitat', fontsize=14, fontweight='bold')
plt.xlabel('Longitude'); plt.ylabel('Latitude')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, linestyle='--', alpha=0.3); plt.tight_layout(); plt.show()

## Worked example: measuring vegetation cover

**How much of a surveyed site is vegetated?** For a multispectral survey we compute NDVI (Normalized Difference Vegetation Index) directly from the red and near-infrared bands of the multispectral orthomosaic, then report the vegetated fraction of the survey. A common first-pass measure for habitat and land-cover work — and it runs off the COG without downloading the full raster.

In [ ]:
# [NEEDS STAC] Compute NDVI from the multispectral orthomosaic and report vegetated fraction.
# Band order for GeoNadir multispectral COGs (see OPEN_DATA_STRUCTURE.md): B1 green, B2 red, B3 rededge, B4 nir.
ms_href = item.assets['multispec_ortho'].href
with rasterio.Env(session=aws_session):
    with rasterio.open(ms_href) as src:
        scale = max(1, max(src.width, src.height) // 2048)
        out_shape = (src.count, src.height // scale, src.width // scale)
        arr = src.read(out_shape=out_shape).astype('float32')

red = arr[1]  # B2
nir = arr[3]  # B4
ndvi = (nir - red) / (nir + red + 1e-9)
valid = np.isfinite(ndvi) & (arr.sum(axis=0) > 0)
vegetated = ndvi > 0.3            # common threshold for healthy vegetation
frac = vegetated[valid].mean() if valid.any() else float('nan')
print(f'Vegetated fraction of survey (NDVI > 0.3): {frac:.1%}')

plt.figure(figsize=(9, 8), dpi=100)
plt.imshow(np.where(valid, ndvi, np.nan), cmap='RdYlGn', vmin=-0.2, vmax=0.9)
plt.colorbar(label='NDVI')
plt.title(f'NDVI — {item_id}', fontsize=13, fontweight='bold')
plt.axis('off'); plt.show()

## Community challenge

**Habitat-scale change and classification from citizen UAV surveys.**

GeoNadir FAIR Data is a growing, globally distributed collection of centimetre-scale UAV surveys tagged with IUCN habitat classes. That combination is unusual: open satellite data is plentiful but coarse; centimetre-scale drone imagery with open licensing and consistent metadata is rare. Some questions we would love the community to take on:

1. **Habitat classification transfer.** Train a land-cover / habitat classifier on the multispectral + RGB orthomosaics and test how well it generalizes across IUCN habitat classes and geographies.
2. **Super-resolution priors for satellite data.** Use the centimetre-scale orthomosaics as ground truth to train or evaluate super-resolution / downscaling of Sentinel-2 or Landsat over the same footprints.
3. **Structure from the DSM/DTM pair.** Where both exist, derive canopy height models and study vegetation structure against the recorded habitat class.

**Getting started.** Filter the STAC catalog by `geonadir:iucn_habitat` to assemble a per-habitat training set. Read overviews from the COGs first (fast, cheap) before committing to full-resolution reads, and treat the `providers` entries as uploader-supplied attribution rather than verified authority records. Start with a toy subset of a handful of surveys — a working small example beats a stalled large one.

---
*Contributions and derived tutorials are welcome — open a pull request against [open-data-examples](https://github.com/Geonadir/open-data-examples) or add your tutorial to the GeoNadir FAIR Data entry on the Registry of Open Data.*